# Validation

This notebook validates that the cleaned dataset, SQL table, analytical SQL queries, and SQL views produce the same results as pandas.

## What is checked

- cleaned CSV vs `rentals_cleaned` table in PostgreSQL
- SQL quality checks vs pandas calculations
- SQL views vs equivalent pandas aggregations

Assumption: the cleaned CSV has already been loaded into PostgreSQL and `sql/views.sql` has already been executed.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import create_engine, text

In [58]:
load_dotenv()

BASE_DIR = Path.cwd().parent
csv_path = BASE_DIR / 'data' / 'rent_cleaned.csv'

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME')

if not all([DB_USER, DB_PASSWORD, DB_NAME]):
    raise ValueError('Set DB_USER, DB_PASSWORD and DB_NAME in .env before running validation.')

engine = create_engine(
    f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

df = pd.read_csv(csv_path)
print(f'CSV shape: {df.shape}')

CSV shape: (197577, 30)


In [ ]:
pd.set_option('display.max_columns', None)

def run_sql(query: str) -> pd.DataFrame:
    return pd.read_sql(text(query), engine)

def normalize_df(frame: pd.DataFrame, sort_by=None, round_cols=None, decimals=2) -> pd.DataFrame:
    result = frame.copy()

    if round_cols:
        for col in round_cols:
            if col in result.columns:
                result[col] = pd.to_numeric(result[col], errors='coerce').round(decimals)

    for col in result.columns:
        if pd.api.types.is_bool_dtype(result[col]):
            result[col] = result[col].astype('boolean')

    if sort_by:
        result = result.sort_values(sort_by, kind='stable').reset_index(drop=True)
    else:
        result = result.reset_index(drop=True)

    return result

def compare_frames(name: str, actual: pd.DataFrame, expected: pd.DataFrame, sort_by=None, round_cols=None, decimals=2, atol=None) -> None:
    actual_norm = normalize_df(actual, sort_by=sort_by, round_cols=round_cols, decimals=decimals)
    expected_norm = normalize_df(expected, sort_by=sort_by, round_cols=round_cols, decimals=decimals)

    actual_norm = actual_norm[expected_norm.columns]

    if list(actual_norm.columns) != list(expected_norm.columns):
        raise AssertionError(
            f"{name}: column mismatch\n"
            f"actual: {list(actual_norm.columns)}\n"
            f"expected: {list(expected_norm.columns)}"
        )

    if actual_norm.shape != expected_norm.shape:
        raise AssertionError(
            f"{name}: shape mismatch: actual={actual_norm.shape}, expected={expected_norm.shape}"
        )

    if atol is None:
        atol = 10 ** (-decimals) + 1e-12 if round_cols else 0

    for col in actual_norm.columns:
        left = actual_norm[col]
        right = expected_norm[col]

        if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
            left_num = pd.to_numeric(left, errors='coerce')
            right_num = pd.to_numeric(right, errors='coerce')

            matches = np.isclose(left_num, right_num, atol=atol, rtol=0, equal_nan=True)

            if not matches.all():
                bad_idx = np.where(~matches)[0][0]
                raise AssertionError(
                    f"{name}: numeric mismatch in column '{col}' at row {bad_idx}: "
                    f"actual={left_num.iloc[bad_idx]}, expected={right_num.iloc[bad_idx]}"
                )
        else:
            matches = (left.fillna('__NA__') == right.fillna('__NA__'))

            if not matches.all():
                bad_idx = np.where(~matches)[0][0]
                raise AssertionError(
                    f"{name}: mismatch in column '{col}' at row {bad_idx}: "
                    f"actual={left.iloc[bad_idx]}, expected={right.iloc[bad_idx]}"
                )

    print(f'[OK] {name}')


def show_check(title: str, actual, expected) -> None:
    check = pd.DataFrame({'actual': [actual], 'expected': [expected], 'match': [actual == expected]}, index=[title])
    display(check)
    if actual != expected:
        raise AssertionError(f'{title}: actual={actual}, expected={expected}')

## 1. Cleaned CSV vs PostgreSQL table

In [60]:
sql_table = run_sql('SELECT * FROM rentals_cleaned')

show_check('row_count', len(sql_table), len(df))
show_check('column_count', sql_table.shape[1], df.shape[1])

csv_columns = sorted(df.columns.tolist())
sql_columns = sorted(sql_table.columns.tolist())
show_check('column_names_match', sql_columns, csv_columns)

,actual,expected,match
row_count,197577,197577,True


,actual,expected,match
column_count,30,30,True


,actual,expected,match
column_names_match,"[balcony, base_rent, cellar, city, cold_rent_p...","[balcony, base_rent, cellar, city, cold_rent_p...",True


In [61]:
numeric_checks = pd.DataFrame([
    {'metric': 'min_total_rent', 'actual': sql_table['total_rent'].min(), 'expected': df['total_rent'].min()},
    {'metric': 'max_total_rent', 'actual': sql_table['total_rent'].max(), 'expected': df['total_rent'].max()},
    {'metric': 'min_living_space', 'actual': sql_table['living_space'].min(), 'expected': df['living_space'].min()},
    {'metric': 'max_living_space', 'actual': sql_table['living_space'].max(), 'expected': df['living_space'].max()},
    {'metric': 'min_price_per_m2', 'actual': sql_table['price_per_m2'].min(), 'expected': df['price_per_m2'].min()},
    {'metric': 'max_price_per_m2', 'actual': sql_table['price_per_m2'].max(), 'expected': df['price_per_m2'].max()}
])
numeric_checks['actual'] = numeric_checks['actual'].round(6)
numeric_checks['expected'] = numeric_checks['expected'].round(6)
numeric_checks['match'] = numeric_checks['actual'] == numeric_checks['expected']
display(numeric_checks)

if not numeric_checks['match'].all():
    raise AssertionError('Base numeric metrics do not match between pandas and SQL table.')

,metric,actual,expected,match
0,min_total_rent,220.000000,220.000000,True
1,max_total_rent,7680.000000,7680.000000,True
2,min_living_space,15.000000,15.000000,True
3,max_living_space,364.000000,364.000000,True
4,min_price_per_m2,2.050000,2.050000,True
5,max_price_per_m2,59.444444,59.444444,True


## 2. SQL quality checks vs pandas

In [62]:
sql_quality = run_sql(
    '''
    SELECT
        COUNT(*) AS total_rows,
        COUNT(heating_type) AS heating_type_not_null,
        COUNT(condition) AS condition_not_null,
        COUNT(interior_qual) AS interior_qual_not_null,
        COUNT(type_of_flat) AS type_of_flat_not_null
    FROM rentals_cleaned
    '''
)

pd_quality = pd.DataFrame([
    {
        'total_rows': len(df),
        'heating_type_not_null': df['heating_type'].notna().sum(),
        'condition_not_null': df['condition'].notna().sum(),
        'interior_qual_not_null': df['interior_qual'].notna().sum(),
        'type_of_flat_not_null': df['type_of_flat'].notna().sum()
    }
])

compare_frames('quality completeness checks', sql_quality, pd_quality)

[OK] quality completeness checks


In [63]:
sql_ranges = run_sql(
    '''
    SELECT
        MIN(total_rent) AS min_total_rent,
        MAX(total_rent) AS max_total_rent,
        MIN(living_space) AS min_living_space,
        MAX(living_space) AS max_living_space,
        MIN(price_per_m2) AS min_price_per_m2,
        MAX(price_per_m2) AS max_price_per_m2
    FROM rentals_cleaned
    '''
)

pd_ranges = pd.DataFrame([
    {
        'min_total_rent': df['total_rent'].min(),
        'max_total_rent': df['total_rent'].max(),
        'min_living_space': df['living_space'].min(),
        'max_living_space': df['living_space'].max(),
        'min_price_per_m2': df['price_per_m2'].min(),
        'max_price_per_m2': df['price_per_m2'].max()
    }
])

compare_frames(
    'quality range checks',
    sql_ranges,
    pd_ranges,
    round_cols=['min_total_rent', 'max_total_rent', 'min_living_space', 'max_living_space', 'min_price_per_m2', 'max_price_per_m2'],
    decimals=6
)

[OK] quality range checks


## Negative property age diagnostic

In [64]:
pd_negative_age = df[df['property_age'] < 0][
    ['city', 'district', 'listing_year', 'year_constructed', 'property_age', 'total_rent', 'living_space', 'no_rooms']
]

sql_negative_age = run_sql(
    '''
    SELECT
        city,
        district,
        listing_year,
        year_constructed,
        property_age,
        total_rent,
        living_space,
        no_rooms
    FROM rentals_cleaned
    WHERE property_age < 0
    ORDER BY property_age, city, district
    '''
)

negative_age_summary = pd.DataFrame([
    {
        'source': 'pandas',
        'negative_property_age_rows': len(pd_negative_age)
    },
    {
        'source': 'sql',
        'negative_property_age_rows': len(sql_negative_age)
    }
])

display(negative_age_summary)

if len(pd_negative_age) > 0:
    print('Pandas examples with negative property_age:')
    display(pd_negative_age.head(10))

if len(sql_negative_age) > 0:
    print('SQL examples with negative property_age:')
    display(sql_negative_age.head(10))

,source,negative_property_age_rows
0,pandas,0
1,sql,0


## 3. SQL views vs pandas equivalents

In [65]:
pd_enriched = df.copy()

pd_enriched['rooms_group'] = np.select(
    [
        pd_enriched['no_rooms'] < 1.5,
        pd_enriched['no_rooms'] < 2.5,
        pd_enriched['no_rooms'] < 3.5,
        pd_enriched['no_rooms'] < 4.5
    ],
    ['1 room', '2 rooms', '3 rooms', '4 rooms'],
    default='5+ rooms'
)

pd_enriched['rooms_group_order'] = np.select(
    [
        pd_enriched['no_rooms'] < 1.5,
        pd_enriched['no_rooms'] < 2.5,
        pd_enriched['no_rooms'] < 3.5,
        pd_enriched['no_rooms'] < 4.5
    ],
    [1, 2, 3, 4],
    default=5
)

pd_enriched['size_group'] = np.select(
    [
        pd_enriched['living_space'] < 40,
        pd_enriched['living_space'] < 60,
        pd_enriched['living_space'] < 80,
        pd_enriched['living_space'] < 100
    ],
    ['Under 40 m2', '40-59 m2', '60-79 m2', '80-99 m2'],
    default='100+ m2'
)

pd_enriched['size_group_order'] = np.select(
    [
        pd_enriched['living_space'] < 40,
        pd_enriched['living_space'] < 60,
        pd_enriched['living_space'] < 80,
        pd_enriched['living_space'] < 100
    ],
    [1, 2, 3, 4],
    default=5
)

pd_enriched['age_group'] = np.select(
    [
        pd_enriched['property_age'] <= 5,
        pd_enriched['property_age'] <= 10,
        pd_enriched['property_age'] <= 20,
        pd_enriched['property_age'] <= 40
    ],
    ['0-5 years', '6-10 years', '11-20 years', '21-40 years'],
    default='41+ years'
)

pd_enriched['age_group_order'] = np.select(
    [
        pd_enriched['property_age'] <= 5,
        pd_enriched['property_age'] <= 10,
        pd_enriched['property_age'] <= 20,
        pd_enriched['property_age'] <= 40
    ],
    [1, 2, 3, 4],
    default=5
)

pd_enriched['condition_group'] = pd_enriched['condition'].fillna('No data')
pd_enriched['interior_qual_group'] = pd_enriched['interior_qual'].fillna('No data')
pd_enriched['construction_status'] = np.where(pd_enriched['newly_const'], 'New build', 'Existing stock')
pd_enriched['price_segment'] = np.select(
    [
        pd_enriched['price_per_m2'] >= 25,
        pd_enriched['price_per_m2'] >= 15,
        pd_enriched['price_per_m2'] >= 10
    ],
    ['Very high price', 'High price', 'Mid price'],
    default='Affordable'
)

In [66]:
sql_enriched = run_sql(
    '''
    SELECT
        city,
        district,
        total_rent,
        living_space,
        no_rooms,
        property_age,
        price_per_m2,
        rooms_group,
        rooms_group_order,
        size_group,
        size_group_order,
        age_group,
        age_group_order,
        condition_group,
        interior_qual_group,
        construction_status,
        price_segment
    FROM vw_rentals_enriched
    '''
)

pd_enriched_subset = pd_enriched[
    [
        'city',
        'district',
        'total_rent',
        'living_space',
        'no_rooms',
        'property_age',
        'price_per_m2',
        'rooms_group',
        'rooms_group_order',
        'size_group',
        'size_group_order',
        'age_group',
        'age_group_order',
        'condition_group',
        'interior_qual_group',
        'construction_status',
        'price_segment'
    ]
]

sql_enriched_counts = (
    normalize_df(
        sql_enriched,
        round_cols=['total_rent', 'living_space', 'no_rooms', 'property_age', 'price_per_m2'],
        decimals=6
    )
    .groupby(list(sql_enriched.columns), dropna=False)
    .size()
    .reset_index(name='row_count')
)

pd_enriched_counts = (
    normalize_df(
        pd_enriched_subset,
        round_cols=['total_rent', 'living_space', 'no_rooms', 'property_age', 'price_per_m2'],
        decimals=6
    )
    .groupby(list(pd_enriched_subset.columns), dropna=False)
    .size()
    .reset_index(name='row_count')
)

compare_frames(
    'vw_rentals_enriched derived columns',
    sql_enriched_counts,
    pd_enriched_counts,
    sort_by=list(sql_enriched.columns) + ['row_count'],
    round_cols=['total_rent', 'living_space', 'no_rooms', 'property_age', 'price_per_m2'],
    decimals=6
)

[OK] vw_rentals_enriched derived columns


In [67]:
sql_city_summary = run_sql('SELECT * FROM vw_city_price_summary')

pd_city_summary = (
    pd_enriched.groupby('city')
    .agg(
        number_of_listings=('city', 'size'),
        avg_price_per_m2=('price_per_m2', 'mean'),
        median_price_per_m2=('price_per_m2', 'median'),
        min_price_per_m2=('price_per_m2', 'min'),
        max_price_per_m2=('price_per_m2', 'max')
    )
    .query('number_of_listings >= 30')
    .reset_index()
)

compare_frames(
    'vw_city_price_summary',
    sql_city_summary,
    pd_city_summary,
    sort_by=['city'],
    round_cols=['avg_price_per_m2', 'median_price_per_m2', 'min_price_per_m2', 'max_price_per_m2']
)

[OK] vw_city_price_summary


In [68]:
overall_median_price = pd_enriched['price_per_m2'].median()

sql_rooms_summary = run_sql('SELECT * FROM vw_rooms_group_summary')

pd_rooms_summary = (
    pd_enriched.groupby(['rooms_group', 'rooms_group_order'])
    .agg(
        number_of_listings=('rooms_group', 'size'),
        median_total_rent=('total_rent', 'median'),
        median_price_per_m2=('price_per_m2', 'median')
    )
    .reset_index()
)
pd_rooms_summary['overall_median_price_per_m2'] = overall_median_price
pd_rooms_summary['price_gap_vs_overall'] = pd_rooms_summary['median_price_per_m2'] - overall_median_price

compare_frames(
    'vw_rooms_group_summary',
    sql_rooms_summary,
    pd_rooms_summary,
    sort_by=['rooms_group_order'],
    round_cols=['median_total_rent', 'median_price_per_m2', 'overall_median_price_per_m2', 'price_gap_vs_overall']
)

[OK] vw_rooms_group_summary


In [69]:
sql_age_summary = run_sql('SELECT * FROM vw_property_age_summary')

pd_age_summary = (
    pd_enriched.groupby(['age_group', 'age_group_order'])
    .agg(
        number_of_listings=('age_group', 'size'),
        median_total_rent=('total_rent', 'median'),
        median_price_per_m2=('price_per_m2', 'median')
    )
    .reset_index()
)
pd_age_summary['overall_median_price_per_m2'] = overall_median_price
pd_age_summary['price_gap_vs_overall'] = pd_age_summary['median_price_per_m2'] - overall_median_price

compare_frames(
    'vw_property_age_summary',
    sql_age_summary,
    pd_age_summary,
    sort_by=['age_group_order'],
    round_cols=['median_total_rent', 'median_price_per_m2', 'overall_median_price_per_m2', 'price_gap_vs_overall']
)

[OK] vw_property_age_summary


In [70]:
sql_amenities_summary = run_sql('SELECT * FROM vw_amenities_summary')

amenities = ['balcony', 'lift', 'has_kitchen', 'garden', 'cellar']
amenity_rows = []

for amenity in amenities:
    grouped = (
        pd_enriched.groupby(amenity)['price_per_m2']
        .agg(['size', 'median'])
        .reset_index()
    )

    with_amenity = grouped[grouped[amenity] == True]
    without_amenity = grouped[grouped[amenity] == False]

    amenity_rows.append({
        'amenity': amenity,
        'listings_with_amenity': with_amenity['size'].iloc[0] if not with_amenity.empty else np.nan,
        'listings_without_amenity': without_amenity['size'].iloc[0] if not without_amenity.empty else np.nan,
        'median_price_with_amenity': with_amenity['median'].iloc[0] if not with_amenity.empty else np.nan,
        'median_price_without_amenity': without_amenity['median'].iloc[0] if not without_amenity.empty else np.nan
    })

pd_amenities_summary = pd.DataFrame(amenity_rows)
pd_amenities_summary['amenity_price_gap'] = (
    pd_amenities_summary['median_price_with_amenity'] - pd_amenities_summary['median_price_without_amenity']
)

compare_frames(
    'vw_amenities_summary',
    sql_amenities_summary,
    pd_amenities_summary,
    sort_by=['amenity'],
    round_cols=['median_price_with_amenity', 'median_price_without_amenity', 'amenity_price_gap']
)

[OK] vw_amenities_summary


## 4. Final result

In [71]:
print('All validation checks passed successfully.')

All validation checks passed successfully.
